In [ ]:
#FULL U NET CODE (MAIN)
import torch
import torch.nn as nn
class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True)

        )

    def forward(self, x):
        return self.conv(x)
class EncoderBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = DoubleConv(
            in_channels,
            out_channels
        )

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

    def forward(self, x):

        features = self.conv(x)

        pooled = self.pool(features)

        return features, pooled
class DecoderBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        self.conv = DoubleConv(
            out_channels * 2,
            out_channels
        )

    def forward(self, x, skip):

        x = self.up(x)

        x = torch.cat([skip, x], dim=1)

        x = self.conv(x)

        return x
class UNet(nn.Module):

    def __init__(self):
        super().__init__()

        # Encoder

        self.enc1 = EncoderBlock(3, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 256)
        self.enc4 = EncoderBlock(256, 512)

        # Bottleneck

        self.bottleneck = DoubleConv(
            512,
            1024
        )

        # Decoder

        self.dec1 = DecoderBlock(
            1024,
            512
        )

        self.dec2 = DecoderBlock(
            512,
            256
        )

        self.dec3 = DecoderBlock(
            256,
            128
        )

        self.dec4 = DecoderBlock(
            128,
            64
        )

        # Final Output

        self.final_conv = nn.Conv2d(
            64,
            1,
            kernel_size=1
        )

    def forward(self, x):

        # Encoder

        f1, p1 = self.enc1(x)

        f2, p2 = self.enc2(p1)

        f3, p3 = self.enc3(p2)

        f4, p4 = self.enc4(p3)

        # Bottleneck

        b = self.bottleneck(p4)

        # Decoder

        d1 = self.dec1(b, f4)

        d2 = self.dec2(d1, f3)

        d3 = self.dec3(d2, f2)

        d4 = self.dec4(d3, f1)

        # Final Output

        out = self.final_conv(d4)

        return out
    
model = UNet()

x = torch.randn(
    1,
    3,
    256,
    256
)

output = model(x)

print("Input Shape :", x.shape)
print("Output Shape:", output.shape)

Input Shape : torch.Size([1, 3, 256, 256])
Output Shape: torch.Size([1, 1, 256, 256])


In [3]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class EncoderBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = DoubleConv(
            in_channels,
            out_channels
        )
        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

    def forward(self, x):
        features = self.conv(x)
        pooled = self.pool(features)
        return features, pooled


class DecoderBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )
        self.conv = DoubleConv(
            out_channels * 2,
            out_channels
        )

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([skip, x], dim=1)
        x = self.conv(x)
        return x


In [4]:
#FINAL MINI U NET CODE
class MiniUNet(nn.Module):

    def __init__(self):
        super().__init__()

        # Encoder

        self.enc1 = EncoderBlock(3, 64)

        self.enc2 = EncoderBlock(64, 128)

        # Bottleneck

        self.bottleneck = DoubleConv(
            128,
            256
        )

        # Decoder

        self.dec1 = DecoderBlock(
            256,
            128
        )

        self.dec2 = DecoderBlock(
            128,
            64
        )

        # Final Output Layer

        self.final_conv = nn.Conv2d(
            64,
            1,
            kernel_size=1
        )

    def forward(self, x):

        # Encoder

        f1, p1 = self.enc1(x)

        f2, p2 = self.enc2(p1)

        # Bottleneck

        b = self.bottleneck(p2)

        # Decoder

        d1 = self.dec1(b, f2)

        d2 = self.dec2(d1, f1)

        # Final Prediction

        out = self.final_conv(d2)

        return out
#test full model
model = MiniUNet()



x = torch.randn(

    1,

    3,

    256,

    256

)



output = model(x)



print("Input Shape :", x.shape)

print("Output Shape:", output.shape)

Input Shape : torch.Size([1, 3, 256, 256])
Output Shape: torch.Size([1, 1, 256, 256])


In [5]:
#decoder block mini  U NET implementation
class DecoderBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        # Upsample image size
        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        # Refine features
        self.conv = DoubleConv(
            out_channels * 2,
            out_channels
        )

    def forward(self, x, skip):

        # Increase spatial size
        x = self.up(x)

        # Join skip connection
        x = torch.cat([skip, x], dim=1)

        # Apply DoubleConv
        x = self.conv(x)

        return x
#test code
decoder = DecoderBlock(256, 128)

x = torch.randn(
    1,
    256,
    64,
    64
)
skip = torch.randn(
    1,
    128,
    128,
    128
)

output = decoder(x, skip)

print(output.shape)

torch.Size([1, 128, 128, 128])


In [6]:
#MINI U NET implementation
# =====================================================
# BOTTLENECK
# =====================================================

bottleneck = DoubleConv(
    128,   # input channels
    256    # output channels
)

# Pass encoder output

b = bottleneck(p2)

print("Bottleneck Shape:", b.shape)

NameError: name 'p2' is not defined

In [ ]:
#MINI U NET implementation

# =====================================================
# DECODER BLOCK
# =====================================================

class DecoderBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        # Upsampling
        # Example:
        # 64x64 -> 128x128

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        # Double Convolution after concatenation

        self.conv = DoubleConv(
            out_channels * 2,
            out_channels
        )

    def forward(self, x, skip):

        # Upsample

        x = self.up(x)

        # Concatenate Skip Connection

        x = torch.cat([skip, x], dim=1)

        # Refine Features

        x = self.conv(x)

        return x

# Decoder

decoder = DecoderBlock(128, 64)
# Simulated encoder output

x = torch.randn(
    1,
    128,
    64,
    64
)

# Simulated skip connection

skip = torch.randn(
    1,
    64,
    128,
    128
)

output = decoder(x, skip)

print("Decoder Output Shape:", output.shape)

Decoder Output Shape: torch.Size([1, 64, 128, 128])


In [ ]:
#MINI U NET implementation

# =====================================================
# U-NET ENCODER (2 LEVELS)
# =====================================================

# Encoder Block 1
# Input Channels  = 3  (RGB Image)
# Output Channels = 64
encoder1 = EncoderBlock(3, 64)

# Encoder Block 2
# Input Channels  = 64 (Output of Encoder 1)
# Output Channels = 128
encoder2 = EncoderBlock(64, 128)

# -----------------------------------------------------
# Create a dummy image for testing
#
# Shape:
# 1   -> Batch Size
# 3   -> RGB Channels
# 256 -> Height
# 256 -> Width
# -----------------------------------------------------
x = torch.randn(1, 3, 256, 256)

# =====================================================
# PASS THROUGH ENCODER LEVEL 1
# =====================================================

# f1 = Features before pooling
# p1 = Output after MaxPool
f1, p1 = encoder1(x)

# =====================================================
# PASS THROUGH ENCODER LEVEL 2
# =====================================================

# p1 becomes input for Encoder 2
# f2 = Features before pooling
# p2 = Output after MaxPool
f2, p2 = encoder2(p1)

# =====================================================
# PRINT SHAPES
# =====================================================

print("----- LEVEL 1 -----")
print("Feature Map Shape :", f1.shape)
print("Pooled Shape      :", p1.shape)

print("\n----- LEVEL 2 -----")
print("Feature Map Shape :", f2.shape)
print("Pooled Shape      :", p2.shape)

----- LEVEL 1 -----
Feature Map Shape : torch.Size([1, 64, 256, 256])
Pooled Shape      : torch.Size([1, 64, 128, 128])

----- LEVEL 2 -----
Feature Map Shape : torch.Size([1, 128, 128, 128])
Pooled Shape      : torch.Size([1, 128, 64, 64])


In [ ]:
#MINI U NET implementation

#construction of encoder
class EncoderBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = DoubleConv(
            in_channels,
            out_channels
        )

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

    def forward(self, x):

        features = self.conv(x)

        pooled = self.pool(features)

        return features, pooled
#testing of encoder
encoder = EncoderBlock(3, 64)

x = torch.randn(1, 3, 256, 256)

features, pooled = encoder(x)

print("Features Shape:", features.shape)
print("Pooled Shape  :", pooled.shape)

Features Shape: torch.Size([1, 64, 256, 256])
Pooled Shape  : torch.Size([1, 64, 128, 128])


In [ ]:
#MINI U NET implementation

#model created ..output verification block
x = torch.randn(1, 3, 256, 256)

block = DoubleConv(3, 64)

y = block(x)

print("Input Shape :", x.shape)
print("Output Shape:", y.shape)

Input Shape : torch.Size([1, 3, 256, 256])
Output Shape: torch.Size([1, 64, 256, 256])


In [ ]:
#MINI U NET implementation

#testing of double convolution
block = DoubleConv(3, 64)

print(block)


DoubleConv(
  (conv): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
  )
)


In [ ]:
import torch
import torch.nn as nn

print(torch.__version__)

2.12.0+cpu


In [ ]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels=3,
            out_channels=16,
            kernel_size=3,
            padding=1
        )

    def forward(self, x):
        return self.conv(x)

model = SimpleModel()

print(model)

SimpleModel(
  (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)


In [ ]:
# Double convolution block
import torch
import torch.nn as nn

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True)

        )

    def forward(self, x):
        return self.conv(x)

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True)

        )

    def forward(self, x):
        return self.conv(x)